In [ ]:
import sys
import os

sys.path.append(
    os.path.abspath(
        os.path.join(os.getcwd(), "..")
    )
)

# Data
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics
from scipy.stats import (
    ttest_ind,
    f_oneway,
    chi2_contingency,
    pearsonr,
    spearmanr
)

# Machine Learning
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

# Database Loader
from db.data_loader import load_data

# Plot Settings
sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
from db.data_loader import load_data

df = load_data()

df.head()

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

df.info()
df.shape

1. Fragestellung:  
Welche Faktoren beeinflussen den Fahrzeugpreis signifikant und eignen sich daher als Einflussvariablen für die Vorhersage des Verkaufspreises?  
H0  
Der durchschnittliche Fahrzeugpreis von Benzin- und Elektrofahrzeugen ist gleich.  
H0: μ_Benzin = μ_Elektro  
H1  
Der durchschnittliche Fahrzeugpreis von Benzin- und Elektrofahrzeugen unterscheidet sich.  
H1: μ_Benzin ≠ μ_Elektro 

In [ ]:
df.groupby("kraftstoff")["preis_euro"].agg(
    ["count", "mean", "std"]
)

### Voraussetzungen für den Zwei-Stichproben-t-Test:
1. **Unabhängigkeit der Stichproben:** Gegeben. Die Preise der Benzinfahrzeuge sind unabhängig von den Preisen der Elektrofahrzeuge, da es sich um separate, eigenständige Verkaufstransaktionen handelt.
2. **Skalierung:** Die abhängige Variable (Preis) ist metrisch skaliert.

In [ ]:
from scipy.stats import ttest_ind

benzin = df[
    df["kraftstoff"] == "Benzin"
]["preis_euro"]

elektro = df[
    df["kraftstoff"] == "Elektro"
]["preis_euro"]

t_stat, p_value = ttest_ind(
    benzin,
    elektro,
    equal_var=False
)

print(f"T-Statistic: {t_stat:.3f}")
print(f"P-Value: {p_value:.5f}")

In [ ]:
alpha = 0.05

if p_value < alpha:
    print("H0 ablehnen")
else:
    print("H0 nicht ablehnen")

Interpretation  
Da p < 0,05 ist, wird die Nullhypothese verworfen. Der Kraftstofftyp hat einen statistisch signifikanten Einfluss auf den Fahrzeugpreis und sollte daher als relevante Einflussvariable für das Machine-Learning-Modell berücksichtigt werden.

2. Korrelationsanalyse  
Fragestellung  
Besteht ein statistisch signifikanter Zusammenhang zwischen Hubraum und Fahrzeugpreis?  
H0  
Zwischen Hubraum und Fahrzeugpreis besteht kein linearer Zusammenhang.  
H0: ρ = 0  
H1  
Zwischen Hubraum und Fahrzeugpreis besteht ein linearer Zusammenhang.  
H1: ρ ≠ 0  

In [ ]:
from scipy.stats import pearsonr

r, p = pearsonr(
    df["hubraum_l"],
    df["preis_euro"]
)

print(f"r = {r:.3f}")
print(f"p = {p:.5f}")

Interpretation  
Der Pearson-Korrelationskoeffizient beträgt r = 0,014 und weist somit auf keinen linearen Zusammenhang zwischen dem Hubraum und dem Fahrzeugpreis hin. Der p-Wert von 0,629 liegt deutlich über dem Signifikanzniveau von α = 0,05. Daher kann die Nullhypothese nicht verworfen werden. Es gibt in diesem Datensatz keine statistisch signifikanten Hinweise darauf, dass der Hubraum den Fahrzeugpreis beeinflusst.